# Chapter 13: Loading and Preprocessing Data with TensorFlow

## Global Imports

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os

# Note: tensorflow_datasets (tfds) and tensorflow_transform (tft)
# usually require separate installation:
!pip install tensorflow-datasets tensorflow-transform
try:
    import tensorflow_datasets as tfds
except ImportError:
    pass # Handle if not installed

# For TFRecords
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.1/994.1 kB 66.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of tensorflow-transform to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 141.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a probl

## The Data API
Deep Learning systems often require datasets that are too large to fit into RAM. TensorFlow’s Data API allows you to create efficient input pipelines that load, parse, and preprocess data from the disk directly. The core concept is the dataset, which represents a sequence of data items.

### Creating a Basic Dataset
You can create a dataset entirely in RAM using from_tensor_slices(), which creates a dataset where each element is a slice of the input tensor.

In [3]:
X = tf.range(10) # tensor containing 0 to 9
dataset = tf.data.Dataset.from_tensor_slices(X)

for item in dataset:
    print(item) # Prints tensors 0 to 9

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


### Chaining Transformations

Dataset methods return new datasets, allowing you to chain transformations. Common methods include:
- repeat(n): Repeats the data n times.
- batch(n): Groups items into batches of size n.
- map(function): Applies a transformation function to each item.
- apply(function): Applies a transformation to the dataset as a whole (e.g., unbatch).
- filter(function): Keeps only items where the function returns True.
- take(n): Creates a dataset with only the first n items.

<p align="left"><img src="../fig/figure13.1.png" width="45%"></p>

In [5]:
import tensorflow as tf

# Make first dataset
dataset = tf.data.Dataset.range(10)

dataset = dataset.repeat(3).batch(7)
dataset = dataset.map(lambda x: x * 2) # Double the items

dataset = dataset.unbatch()

# Now filter works in units
dataset = dataset.filter(lambda x: x < 10)

for item in dataset.take(3):
    print(item.numpy())

0
2
4


### Shuffling the Data
Gradient Descent requires independent and identically distributed instances. The shuffle(buffer_size) method loads items into a buffer and samples them randomly. The buffer_size is crucial; if it's too small, the shuffling won't be effective.

In [6]:
dataset = tf.data.Dataset.range(10).repeat(3)
dataset = dataset.shuffle(buffer_size=5, seed=42).batch(7)
for item in dataset:
    print(item)

tf.Tensor([0 2 3 6 7 9 4], shape=(7,), dtype=int64)
tf.Tensor([5 0 1 1 8 6 5], shape=(7,), dtype=int64)
tf.Tensor([4 8 7 1 2 3 0], shape=(7,), dtype=int64)
tf.Tensor([5 4 2 7 8 9 9], shape=(7,), dtype=int64)
tf.Tensor([3 6], shape=(2,), dtype=int64)


For large datasets that don't fit in memory, simple shuffling isn't enough. You should:
1. Shuffle the source files themselves.
2. Use list_files to shuffle filenames.
3. Use interleave to read from multiple files simultaneously.
Interleaving: This reads from multiple files at once, cycling through them to yield lines. This avoids having instances from the same file clumped together.

In [7]:
# Assuming train_filepaths is a list of file paths like ["train_01.csv", "train_02.csv", ...]
# filepath_dataset = tf.data.Dataset.list_files(train_filepaths, seed=42)

# n_readers = 5
# dataset = filepath_dataset.interleave(
#     lambda filepath: tf.data.TextLineDataset(filepath).skip(1), # skip header
#     cycle_length=n_readers
# )

## Preprocessing the Data
The data loaded (e.g., from CSVs) is often raw text strings. You need to parse and normalize it.

### Parsing CSV Lines
The tf.io.decode_csv function parses a line. You must provide record_defaults to specify the default value and the data type for each column.

In [8]:
# Assuming we have precomputed stats
X_mean, X_std = [0.] * 8, [1.] * 8 # Dummies for example
n_inputs = 8

def preprocess(line):
    defs = [0.] * n_inputs + [tf.constant([], dtype=tf.float32)]
    fields = tf.io.decode_csv(line, record_defaults=defs)
    x = tf.stack(fields[:-1])
    y = tf.stack(fields[-1:])
    return (x - X_mean) / X_std, y

# Test the function
preprocess(b'4.2083,44.0,5.3232,0.9171,846.0,2.3370,37.47,-122.2,2.782')

(<tf.Tensor: shape=(8,), dtype=float32, numpy=
 array([   4.2083,   44.    ,    5.3232,    0.9171,  846.    ,    2.337 ,
          37.47  , -122.2   ], dtype=float32)>,
 <tf.Tensor: shape=(1,), dtype=float32, numpy=array([2.782], dtype=float32)>)

### Putting It All Together
We can wrap the loading, preprocessing, shuffling, and batching into a reusable helper function. Crucially, the .prefetch(1) method is added at the end. This allows the CPU to prepare the next batch while the GPU is training on the current one, maximizing hardware utilization.

<p align="left"><img src="../fig/figure13.2.png" width="45%"></p>

In [9]:
def csv_reader_dataset(filepaths, repeat=1, n_readers=5,
                       n_read_threads=None, shuffle_buffer_size=10000,
                       n_parse_threads=5, batch_size=32):
    dataset = tf.data.Dataset.list_files(filepaths)
    dataset = dataset.interleave(
        lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
        cycle_length=n_readers, num_parallel_calls=n_read_threads)
    dataset = dataset.map(preprocess, num_parallel_calls=n_parse_threads)
    dataset = dataset.shuffle(shuffle_buffer_size).repeat(repeat)
    return dataset.batch(batch_size).prefetch(1)

You can then pass this dataset directly to Keras methods like model.fit().

In [10]:
# train_set = csv_reader_dataset(train_filepaths)
# valid_set = csv_reader_dataset(valid_filepaths)
# model.fit(train_set, epochs=10, validation_data=valid_set)

## The TFRecord Format
TFRecord is TensorFlow’s preferred binary format for storing large datasets. It consists of a sequence of binary records.

### Writing and Reading TFRecords
You use TFRecordWriter to write and TFRecordDataset to read.

In [11]:
# Writing
with tf.io.TFRecordWriter("my_data.tfrecord") as f:
    f.write(b"This is the first record")
    f.write(b"And this is the second record")

# Reading
filepaths = ["my_data.tfrecord"]
dataset = tf.data.TFRecordDataset(filepaths)
for item in dataset:
    print(item)

tf.Tensor(b'This is the first record', shape=(), dtype=string)
tf.Tensor(b'And this is the second record', shape=(), dtype=string)


You can also compress files by setting compression_type="GZIP" in TFRecordOptions.

### Protocol Buffers
TFRecords usually contain serialized Protocol Buffers (protobufs). The standard protobuf used is Example, which contains a list of named Features.

### Creating and Writing an Example:

In [12]:
# Create an Example
person_example = Example(
    features=Features(
        feature={
            "name": Feature(bytes_list=BytesList(value=[b"Alice"])),
            "id": Feature(int64_list=Int64List(value=[123])),
            "emails": Feature(bytes_list=BytesList(value=[b"a@b.com", b"c@d.com"]))
        }))

# Write serialized Example to file
with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    f.write(person_example.SerializeToString())

### Parsing Examples:
 To read them back, you define a description dictionary and use tf.io.parse_single_example.

In [13]:
feature_description = {
    "name": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails": tf.io.VarLenFeature(tf.string),
}

for serialized_example in tf.data.TFRecordDataset(["my_contacts.tfrecord"]):
    parsed_example = tf.io.parse_single_example(serialized_example, feature_description)
    print(parsed_example["name"])

tf.Tensor(b'Alice', shape=(), dtype=string)


For hierarchical data (lists of lists, like documents containing sentences containing words), you can use the SequenceExample protobuf.

## Preprocessing Input Features\
Neural networks need numerical inputs. Categorical and text features must be encoded.

### Encoding Categorical Features (One-Hot)
If there are few categories (e.g., < 10), use one-hot encoding. You map categories to indices using a Lookup Table.

In [14]:
vocab = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]
indices = tf.range(len(vocab), dtype=tf.int64)
table_init = tf.lookup.KeyValueTensorInitializer(vocab, indices)
num_oov_buckets = 2
table = tf.lookup.StaticVocabularyTable(table_init, num_oov_buckets)

# Convert categories to indices
categories = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND"])
cat_indices = table.lookup(categories)
print(cat_indices) # [3, 5, 1, 1] (DESERT is unknown -> index 5)

# Convert to One-Hot
cat_one_hot = tf.one_hot(cat_indices, depth=len(vocab) + num_oov_buckets)

tf.Tensor([3 5 1 1], shape=(4,), dtype=int64)


### Embeddings
If categories are numerous (> 50), one-hot vectors become too sparse. Embeddings are dense, trainable vectors representing categories. Similar categories learn to be close in the embedding space (e.g., "King" - "Man" + "Woman" $\approx$ "Queen").

<p align="left"><img src="../fig/figure13.4.png" width="45%"></p>
<p align="left"><img src="../fig/figure13.5.png" width="45%"></p>

### Manual Implementation:

In [15]:
embedding_dim = 2
embed_init = tf.random.uniform([len(vocab) + num_oov_buckets, embedding_dim])
embedding_matrix = tf.Variable(embed_init)

# Look up embeddings
cat_indices = table.lookup(categories)
tf.nn.embedding_lookup(embedding_matrix, cat_indices)

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[0.63693357, 0.73770165],
       [0.6369661 , 0.3238914 ],
       [0.26222837, 0.3459443 ],
       [0.26222837, 0.3459443 ]], dtype=float32)>

### Keras Layer Implementation:

In [16]:
embedding = keras.layers.Embedding(input_dim=len(vocab) + num_oov_buckets,
                                   output_dim=embedding_dim)
print(embedding(cat_indices))

tf.Tensor(
[[-0.02704263  0.02176506]
 [-0.01127089 -0.00777583]
 [ 0.02292174  0.04774283]
 [ 0.02292174  0.04774283]], shape=(4, 2), dtype=float32)


## Keras Preprocessing Layers
Keras provides standard preprocessing layers that can be included directly in the model. They handle the logic of adapting to data (calculating means, vocabularies, etc.).

- Normalization: Replaces standard scaling.
- TextVectorization: Converts text to word indices or bag-of-words/TF-IDF.
- Discretization: Bins continuous data (e.g., prices into low, medium, high).

### Custom Standardization Layer Example:
You can create a custom layer that mimics StandardScaler.

In [17]:
class Standardization(keras.layers.Layer):
    def adapt(self, data_sample):
        self.means_ = np.mean(data_sample, axis=0, keepdims=True)
        self.stds_ = np.std(data_sample, axis=0, keepdims=True)

    def call(self, inputs):
        return (inputs - self.means_) / (self.stds_ + keras.backend.epsilon())

# Usage
# std_layer = Standardization()
# std_layer.adapt(data_sample)
# model.add(std_layer)

## TF Transform
Preprocessing on-the-fly (during training) works well but can be slow. Preprocessing ahead of time speeds up training but can lead to training/serving skew (mismatches between training logic and app logic).

TF Transform solves this by letting you define the preprocessing once. It runs efficiently (using Apache Beam) to preprocess the training data and exports a TensorFlow Function that gets added to your model. This ensures the deployed model performs the exact same preprocessing.

In [18]:
# Example pseudo-code for a TF Transform preprocessing function
# import tensorflow_transform as tft

# def preprocess(inputs):
#     median_age = inputs["housing_median_age"]
#     standardized_age = tft.scale_to_z_score(median_age)
#     ocean_proximity = inputs["ocean_proximity"]
#     ocean_proximity_id = tft.compute_and_apply_vocabulary(ocean_proximity)
#     return {
#         "standardized_median_age": standardized_age,
#         "ocean_proximity_id": ocean_proximity_id
#     }

## The TensorFlow Datasets (TFDS) Project
TFDS provides a collection of ready-to-use datasets (MNIST, ImageNet, etc.).

In [19]:
# Ensure tfds is imported
if 'tensorflow_datasets' in globals():
    dataset = tfds.load(name="mnist", batch_size=32, as_supervised=True)
    mnist_train = dataset["train"].prefetch(1)

    # Ready to train
    model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(10, activation="softmax")
    ])
    model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd")
    # model.fit(mnist_train, epochs=5)